## Basic Imports

In [1]:
import os
import finnhub
import pandas as pd
import json
import duckdb as dd
import requests
import bs4 as bs
from bs4 import BeautifulSoup
from functools import reduce
from dotenv import load_dotenv, find_dotenv
import time
## from datetime import datetime
## from urllib.parse import quote_plus
## import lxml

load_dotenv(find_dotenv())
api_key = os.getenv('FINNHUB_API_KEY')
DB_PATH = "finnhub.duckdb"
SCHEMA_NAME = "my_schema"
delay_seconds = 1
finnhub_client = finnhub.Client(api_key=api_key)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Drop all old tables from main schema
conn = dd.connect(DB_PATH)
conn.execute(f'DROP SCHEMA IF EXISTS {SCHEMA_NAME} CASCADE;')
conn.execute(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME};')
conn.close()

def save_db_table_from_json(db_path: str = DB_PATH, schema_name: str = SCHEMA_NAME, table_name: str = None, json_data: dict = None):
    if db_path is None or schema_name is None or table_name is None or json_data is None:
        print("Error: All of db_path, schema_name, table_name, and json_data must be provided!")
        pass
    else:
        json_data = json.dumps(json_data)  # Convert dict to JSON string
        ## json_data = json_data.replace('""', '"N/A"').replace('\"','')  # Escape single quotes for SQL
        conn = dd.connect(db_path)
        conn.execute(
            f'''
            CREATE TABLE IF NOT EXISTS {schema_name}.{table_name} AS (
                SELECT DISTINCT *
                FROM read_json({json_data})
            );

            INSERT INTO {schema_name}.{table_name} 
            SELECT DISTINCT *
            FROM read_json({json_data})
            WHERE NOT EXISTS (
                SELECT 1
                FROM {schema_name}.{table_name} AS existing
                WHERE existing.url = read_json({json_data}).url
            );
            '''
        )
        conn.close()

def save_db_table_from_df(db_path: str = DB_PATH, table_name: str = None, df: pd.DataFrame = None):
    if db_path is None or table_name is None or df is None:
        print("Error: All of db_path, table_name, and df must be provided!")
    else:
        dd.register("df", df)
        conn = dd.connect(db_path)
        conn.execute(
            f'''
            DROP TABLE IF EXISTS {table_name};

            CREATE TABLE IF NOT EXISTS {table_name} AS (
                SELECT DISTINCT *
                FROM df
            );
            '''
        )
        conn.close()

def normalize_json_dataframe(df: pd.DataFrame = None):
    if df is None:
        print("Error: DataFrame must be provided!")
        return None
    else:
        result_dataframes = {}
        
        for idx, row in df.iterrows():
            row_normalized_dfs = []
            
            for col in df.columns:
                cell_value = row[col]
                
                # Parse JSON if it's a string
                if isinstance(cell_value, str):
                    try:
                        json_data = json.loads(cell_value)
                    except json.JSONDecodeError:
                        continue  # Skip invalid JSON
                else:
                    json_data = cell_value
                
                # Normalize if it's a dict or list of dicts
                if isinstance(json_data, (dict, list)):
                    try:
                        normalized_df = pd.json_normalize(json_data)
                        normalized_df['original_index'] = idx  # Add original index to the normalized DataFrame
                        rename_cols = [i for i in normalized_df.columns if i == 'v']
                        for i in rename_cols:
                            normalized_df.rename(columns={i: f"{col}_{i}"}, inplace=True)
                        row_normalized_dfs.append(normalized_df)
                    except Exception:
                        continue  # Skip if normalization fails
            
            # Combine all normalized DataFrames for this row
            if row_normalized_dfs:
                combined_df = reduce(lambda left, right: pd.merge(left, right, on=['period', 'original_index'], how='outer'), row_normalized_dfs)
                ## combined_df.set_index('original_index', inplace=True)
                result_dataframes[idx] = combined_df
        
        return result_dataframes

## SCRAPE S&P 500 TICKERS

In [2]:
s_p_500_ticker_url = 'http://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
wiki_headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Accept-Encoding': 'gzip, deflate, br',
        'Origin': 'https://en.wikipedia.org',
        'Referer': 'https://en.wikipedia.org/',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Site': 'same-origin',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-User': '?1',
        'Sec-Fetch-Dest': 'document',
        'Sec-CH-UA': '"Google Chrome";v="146", "Chromium";v="146", "Not=A?Brand";v="24"',
        'Sec-CH-UA-Mobile': '?0',
        'Sec-CH-UA-Platform': '"Windows"',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    }
resp = requests.get(s_p_500_ticker_url, headers=wiki_headers)
soup = bs.BeautifulSoup(resp.text, 'html.parser')
table = soup.select_one('table[id*="constituents"]')
tickers = [row.find_all('td')[0].text.strip() for row in table.find_all('tr')[1:]]
tickers = tickers[0:5]

## Free Plan Endpoints

##### Company News

In [ ]:
# From JSON version
table_name = 'company_news'
for ticker in tickers:
    save_db_table_from_json(schema_name = SCHEMA_NAME, table_name=table_name, json_data=finnhub_client.company_news(ticker, _from=pd.to_datetime("today").date(), to=pd.to_datetime("today").date()))
    time.sleep(delay_seconds)

conn = dd.connect(DB_PATH)  # Open the persistent database
result = conn.sql(f'''
                 SELECT *
                 FROM {SCHEMA_NAME}.{table_name}
                 ;
                 '''
                 ).df()
conn.close()
result

##### Company Basic Financials

In [ ]:
# Basic financials
## print(finnhub_client.company_basic_financials(test_symbol, 'all'))
df = pd.DataFrame(finnhub_client.company_basic_financials(test_symbol, 'all'))
annual_and_quarterly_pre_df = df.copy(deep=True)

annual_and_quarterly_pre_df.index.name = 'index'
annual_and_quarterly_pre_df.reset_index(inplace=True)

annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df[annual_and_quarterly_pre_df['index'] == 'annual']).iloc[-2:,:].set_index('symbol') ## .reset_index()
## annual_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['annual']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
annual_df = pd.json_normalize(annual_pre_df['series']).reset_index()

quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df[annual_and_quarterly_pre_df['index'] == 'quarterly']).iloc[-2:,:].set_index('symbol') ## .reset_index()
## quarterly_pre_df = pd.DataFrame(annual_and_quarterly_pre_df.loc['quarterly']).iloc[-2:,:].T.set_index('symbol') ## .reset_index()
quarterly_df = pd.json_normalize(quarterly_pre_df['series']).reset_index()

df = df.drop(['annual','quarterly'], axis=0)
df = df.drop(columns=['metricType','series']).reset_index()

quarterly_df.head()

In [ ]:
duckdb.register("df_view", df)
res = duckdb.query("SELECT * FROM df_view WHERE index == '10DayAverageTradingVolume'").to_df()
res.head()

In [ ]:
## annual_json_parsed_dfs = normalize_json_dataframe(annual_df)
## duckdb.register("annual_json_parsed_df_view", annual_json_parsed_dfs)
duckdb.register("annual_df_view", annual_df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(bookValue), '$.period') AS DATE) AS bookValue_period, \
                   CAST(json_extract(unnest(bookValue), '$.v') AS FLOAT) AS bookValue_value, \
                   CAST(json_extract(unnest(cashRatio), '$.period') AS DATE) AS cashRatio_period, \
                   CAST(json_extract(unnest(cashRatio), '$.v') AS FLOAT) AS cashRatio_value, \
                   CAST(json_extract(unnest(currentRatio), '$.period') AS DATE) AS currentRatio_period, \
                   CAST(json_extract(unnest(currentRatio), '$.v') AS FLOAT) AS currentRatio_value, \
                   CAST(json_extract(unnest(ebitPerShare), '$.period') AS DATE) AS ebitPerShare_period, \
                   CAST(json_extract(unnest(ebitPerShare), '$.v') AS FLOAT) AS ebitPerShare_value, \
                   CAST(json_extract(unnest(eps), '$.period') AS DATE) AS eps_period, \
                   CAST(json_extract(unnest(eps), '$.v') AS FLOAT) AS eps_value, \
                   CAST(json_extract(unnest(ev), '$.period') AS DATE) AS ev_period, \
                   CAST(json_extract(unnest(ev), '$.v') AS FLOAT) AS ev_value, \
                   CAST(json_extract(unnest(evEbitda), '$.period') AS DATE) AS evEbitda_period, \
                   CAST(json_extract(unnest(evEbitda), '$.v') AS FLOAT) AS evEbitda_value, \
                   CAST(json_extract(unnest(evRevenue), '$.period') AS DATE) AS evRevenue_period, \
                   CAST(json_extract(unnest(evRevenue), '$.v') AS FLOAT) AS evRevenue_value, \
                   CAST(json_extract(unnest(fcfMargin), '$.period') AS DATE) AS fcfMargin_period, \
                   CAST(json_extract(unnest(fcfMargin), '$.v') AS FLOAT) AS fcfMargin_value, \
                   CAST(json_extract(unnest(grossMargin), '$.period') AS DATE) AS grossMargin_period, \
                   CAST(json_extract(unnest(grossMargin), '$.v') AS FLOAT) AS grossMargin_value, \
                   CAST(json_extract(unnest(inventoryTurnover), '$.period') AS DATE) AS inventoryTurnover_period, \
                   CAST(json_extract(unnest(inventoryTurnover), '$.v') AS FLOAT) AS inventoryTurnover_value, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.period') AS DATE) AS longtermDebtTotalAsset_period, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.v') AS FLOAT) AS longtermDebtTotalAsset_value, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.period') AS DATE) AS longtermDebtTotalCapital_period, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.v') AS FLOAT) AS longtermDebtTotalCapital_value, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.period') AS DATE) AS longtermDebtTotalEquity_period, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.v') AS FLOAT) AS longtermDebtTotalEquity_value, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.period') AS DATE) AS netDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.v') AS FLOAT) AS netDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.period') AS DATE) AS netDebtToTotalEquity_period, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.v') AS FLOAT) AS netDebtToTotalEquity_value, \
                   CAST(json_extract(unnest(netMargin), '$.period') AS DATE) AS netMargin_period, \
                   CAST(json_extract(unnest(netMargin), '$.v') AS FLOAT) AS netMargin_value, \
                   CAST(json_extract(unnest(operatingMargin), '$.period') AS DATE) AS operatingMargin_period, \
                   CAST(json_extract(unnest(operatingMargin), '$.v') AS FLOAT) AS operatingMargin_value, \
                   CAST(json_extract(unnest(payoutRatio), '$.period') AS DATE) AS payoutRatio_period, \
                   CAST(json_extract(unnest(payoutRatio), '$.v') AS FLOAT) AS payoutRatio_value, \
                   CAST(json_extract(unnest(pb), '$.period') AS DATE) AS pb_period, \
                   CAST(json_extract(unnest(pb), '$.v') AS FLOAT) AS pb_value, \
                   CAST(json_extract(unnest(pe), '$.period') AS DATE) AS pe_period, \
                   CAST(json_extract(unnest(pe), '$.v') AS FLOAT) AS pe_value, \
                   CAST(json_extract(unnest(pfcf), '$.period') AS DATE) AS pfcf_period, \
                   CAST(json_extract(unnest(pfcf), '$.v') AS FLOAT) AS pfcf_value, \
                   CAST(json_extract(unnest(pretaxMargin), '$.period') AS DATE) AS pretaxMargin_period, \
                   CAST(json_extract(unnest(pretaxMargin), '$.v') AS FLOAT) AS pretaxMargin_value, \
                   CAST(json_extract(unnest(ps), '$.period') AS DATE) AS ps_period, \
                   CAST(json_extract(unnest(ps), '$.v') AS FLOAT) AS ps_value, \
                   CAST(json_extract(unnest(ptbv), '$.period') AS DATE) AS ptbv_period, \
                   CAST(json_extract(unnest(ptbv), '$.v') AS FLOAT) AS ptbv_value, \
                   CAST(json_extract(unnest(quickRatio), '$.period') AS DATE) AS quickRatio_period, \
                   CAST(json_extract(unnest(quickRatio), '$.v') AS FLOAT) AS quickRatio_value, \
                   CAST(json_extract(unnest(receivablesTurnover), '$.period') AS DATE) AS receivablesTurnover_period, \
                   CAST(json_extract(unnest(receivablesTurnover), '$.v') AS FLOAT) AS receivablesTurnover_value, \
                   CAST(json_extract(unnest(roa), '$.period') AS DATE) AS roa_period, \
                   CAST(json_extract(unnest(roa), '$.v') AS FLOAT) AS roa_value, \
                   CAST(json_extract(unnest(roe), '$.period') AS DATE) AS roe_period, \
                   CAST(json_extract(unnest(roe), '$.v') AS FLOAT) AS roe_value, \
                   CAST(json_extract(unnest(roic), '$.period') AS DATE) AS roic_period, \
                   CAST(json_extract(unnest(roic), '$.v') AS FLOAT) AS roic_value, \
                   CAST(json_extract(unnest(rotc), '$.period') AS DATE) AS rotc_period, \
                   CAST(json_extract(unnest(rotc), '$.v') AS FLOAT) AS rotc_value, \
                   CAST(json_extract(unnest(salesPerShare), '$.period') AS DATE) AS salesPerShare_period, \
                   CAST(json_extract(unnest(salesPerShare), '$.v') AS FLOAT) AS salesPerShare_value, \
                   CAST(json_extract(unnest(sgaToSale), '$.period') AS DATE) AS sgaToSale_period, \
                   CAST(json_extract(unnest(sgaToSale), '$.v') AS FLOAT) AS sgaToSale_value, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.period') AS DATE) AS tangibleBookValue_period, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.v') AS FLOAT) AS tangibleBookValue_value, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.period') AS DATE) AS totalDebtToEquity_period, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.v') AS FLOAT) AS totalDebtToEquity_value, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.period') AS DATE) AS totalDebtToTotalAsset_period, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.v') AS FLOAT) AS totalDebtToTotalAsset_value, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.period') AS DATE) AS totalDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.v') AS FLOAT) AS totalDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(totalRatio), '$.period') AS DATE) AS totalRatio_period, \
                   CAST(json_extract(unnest(totalRatio), '$.v') AS FLOAT) AS totalRatio_value, \
                   symbol FROM annual_df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.head()


In [ ]:
## quarterly_json_parsed_dfs = normalize_json_dataframe(quarterly_df)
## duckdb.register("quarterly_json_parsed_df_view", quarterly_json_parsed_dfs)
duckdb.register("quarterly_df_view", quarterly_df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(assetTurnoverTTM), '$.period') AS DATE) AS assetTurnoverTTM_period, \
                   CAST(json_extract(unnest(assetTurnoverTTM), '$.v') AS FLOAT) AS assetTurnoverTTM_value, \
                   CAST(json_extract(unnest(bookValue), '$.period') AS DATE) AS bookValue_period, \
                   CAST(json_extract(unnest(bookValue), '$.v') AS FLOAT) AS bookValue_value, \
                   CAST(json_extract(unnest(cashRatio), '$.period') AS DATE) AS cashRatio_period, \
                   CAST(json_extract(unnest(cashRatio), '$.v') AS FLOAT) AS cashRatio_value, \
                   CAST(json_extract(unnest(currentRatio), '$.period') AS DATE) AS currentRatio_period, \
                   CAST(json_extract(unnest(currentRatio), '$.v') AS FLOAT) AS currentRatio_value, \
                   CAST(json_extract(unnest(ebitPerShare), '$.period') AS DATE) AS ebitPerShare_period, \
                   CAST(json_extract(unnest(ebitPerShare), '$.v') AS FLOAT) AS ebitPerShare_value, \
                   CAST(json_extract(unnest(eps), '$.period') AS DATE) AS eps_period, \
                   CAST(json_extract(unnest(eps), '$.v') AS FLOAT) AS eps_value, \
                   CAST(json_extract(unnest(ev), '$.period') AS DATE) AS ev_period, \
                   CAST(json_extract(unnest(ev), '$.v') AS FLOAT) AS ev_value, \
                   CAST(json_extract(unnest(evEbitdaTTM), '$.period') AS DATE) AS evEbitdaTTM_period, \
                   CAST(json_extract(unnest(evEbitdaTTM), '$.v') AS FLOAT) AS evEbitdaTTM_value, \
                   CAST(json_extract(unnest(evRevenueTTM), '$.period') AS DATE) AS evRevenueTTM_period, \
                   CAST(json_extract(unnest(evRevenueTTM), '$.v') AS FLOAT) AS evRevenueTTM_value, \
                   CAST(json_extract(unnest(fcfMargin), '$.period') AS DATE) AS fcfMargin_period, \
                   CAST(json_extract(unnest(fcfMargin), '$.v') AS FLOAT) AS fcfMargin_value, \
                   CAST(json_extract(unnest(fcfPerShareTTM), '$.period') AS DATE) AS fcfPerShareTTM_period, \
                   CAST(json_extract(unnest(fcfPerShareTTM), '$.v') AS FLOAT) AS fcfPerShareTTM_value, \
                   CAST(json_extract(unnest(grossMargin), '$.period') AS DATE) AS grossMargin_period, \
                   CAST(json_extract(unnest(grossMargin), '$.v') AS FLOAT) AS grossMargin_value, \
                   CAST(json_extract(unnest(inventoryTurnoverTTM), '$.period') AS DATE) AS inventoryTurnoverTTM_period, \
                   CAST(json_extract(unnest(inventoryTurnoverTTM), '$.v') AS FLOAT) AS inventoryTurnoverTTM_value, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.period') AS DATE) AS longtermDebtTotalAsset_period, \
                   CAST(json_extract(unnest(longtermDebtTotalAsset), '$.v') AS FLOAT) AS longtermDebtTotalAsset_value, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.period') AS DATE) AS longtermDebtTotalCapital_period, \
                   CAST(json_extract(unnest(longtermDebtTotalCapital), '$.v') AS FLOAT) AS longtermDebtTotalCapital_value, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.period') AS DATE) AS longtermDebtTotalEquity_period, \
                   CAST(json_extract(unnest(longtermDebtTotalEquity), '$.v') AS FLOAT) AS longtermDebtTotalEquity_value, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.period') AS DATE) AS netDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(netDebtToTotalCapital), '$.v') AS FLOAT) AS netDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.period') AS DATE) AS netDebtToTotalEquity_period, \
                   CAST(json_extract(unnest(netDebtToTotalEquity), '$.v') AS FLOAT) AS netDebtToTotalEquity_value, \
                   CAST(json_extract(unnest(netMargin), '$.period') AS DATE) AS netMargin_period, \
                   CAST(json_extract(unnest(netMargin), '$.v') AS FLOAT) AS netMargin_value, \
                   CAST(json_extract(unnest(operatingMargin), '$.period') AS DATE) AS operatingMargin_period, \
                   CAST(json_extract(unnest(operatingMargin), '$.v') AS FLOAT) AS operatingMargin_value, \
                   CAST(json_extract(unnest(payoutRatioTTM), '$.period') AS DATE) AS payoutRatioTTM_period, \
                   CAST(json_extract(unnest(payoutRatioTTM), '$.v') AS FLOAT) AS payoutRatioTTM_value, \
                   CAST(json_extract(unnest(pb), '$.period') AS DATE) AS pb_period, \
                   CAST(json_extract(unnest(pb), '$.v') AS FLOAT) AS pb_value, \
                   CAST(json_extract(unnest(peTTM), '$.period') AS DATE) AS peTTM_period, \
                   CAST(json_extract(unnest(peTTM), '$.v') AS FLOAT) AS peTTM_value, \
                   CAST(json_extract(unnest(pfcfTTM), '$.period') AS DATE) AS pfcfTTM_period, \
                   CAST(json_extract(unnest(pfcfTTM), '$.v') AS FLOAT) AS pfcfTTM_value, \
                   CAST(json_extract(unnest(pretaxMargin), '$.period') AS DATE) AS pretaxMargin_period, \
                   CAST(json_extract(unnest(pretaxMargin), '$.v') AS FLOAT) AS pretaxMargin_value, \
                   CAST(json_extract(unnest(psTTM), '$.period') AS DATE) AS psTTM_period, \
                   CAST(json_extract(unnest(psTTM), '$.v') AS FLOAT) AS psTTM_value, \
                   CAST(json_extract(unnest(ptbv), '$.period') AS DATE) AS ptbv_period, \
                   CAST(json_extract(unnest(ptbv), '$.v') AS FLOAT) AS ptbv_value, \
                   CAST(json_extract(unnest(quickRatio), '$.period') AS DATE) AS quickRatio_period, \
                   CAST(json_extract(unnest(quickRatio), '$.v') AS FLOAT) AS quickRatio_value, \
                   CAST(json_extract(unnest(receivablesTurnoverTTM), '$.period') AS DATE) AS receivablesTurnoverTTM_period, \
                   CAST(json_extract(unnest(receivablesTurnoverTTM), '$.v') AS FLOAT) AS receivablesTurnoverTTM_value, \
                   CAST(json_extract(unnest(roaTTM), '$.period') AS DATE) AS roaTTM_period, \
                   CAST(json_extract(unnest(roaTTM), '$.v') AS FLOAT) AS roaTTM_value, \
                   CAST(json_extract(unnest(roeTTM), '$.period') AS DATE) AS roeTTM_period, \
                   CAST(json_extract(unnest(roeTTM), '$.v') AS FLOAT) AS roeTTM_value, \
                   CAST(json_extract(unnest(roicTTM), '$.period') AS DATE) AS roicTTM_period, \
                   CAST(json_extract(unnest(roicTTM), '$.v') AS FLOAT) AS roicTTM_value, \
                   CAST(json_extract(unnest(rotcTTM), '$.period') AS DATE) AS rotcTTM_period, \
                   CAST(json_extract(unnest(rotcTTM), '$.v') AS FLOAT) AS rotcTTM_value, \
                   CAST(json_extract(unnest(salesPerShare), '$.period') AS DATE) AS salesPerShare_period, \
                   CAST(json_extract(unnest(salesPerShare), '$.v') AS FLOAT) AS salesPerShare_value, \
                   CAST(json_extract(unnest(sgaToSale), '$.period') AS DATE) AS sgaToSale_period, \
                   CAST(json_extract(unnest(sgaToSale), '$.v') AS FLOAT) AS sgaToSale_value, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.period') AS DATE) AS tangibleBookValue_period, \
                   CAST(json_extract(unnest(tangibleBookValue), '$.v') AS FLOAT) AS tangibleBookValue_value, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.period') AS DATE) AS totalDebtToEquity_period, \
                   CAST(json_extract(unnest(totalDebtToEquity), '$.v') AS FLOAT) AS totalDebtToEquity_value, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.period') AS DATE) AS totalDebtToTotalAsset_period, \
                   CAST(json_extract(unnest(totalDebtToTotalAsset), '$.v') AS FLOAT) AS totalDebtToTotalAsset_value, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.period') AS DATE) AS totalDebtToTotalCapital_period, \
                   CAST(json_extract(unnest(totalDebtToTotalCapital), '$.v') AS FLOAT) AS totalDebtToTotalCapital_value, \
                   CAST(json_extract(unnest(totalRatio), '$.period') AS DATE) AS totalRatio_period, \
                   CAST(json_extract(unnest(totalRatio), '$.v') AS FLOAT) AS totalRatio_value, \
                   symbol FROM quarterly_df_view").to_df()  ##WHERE period == '1989-06-30'").to_df()
res.head()

##### Earnings Surprises

In [ ]:
# Earnings surprises
## print(finnhub_client.company_earnings(test_symbol, limit=5))
df = pd.DataFrame(finnhub_client.company_earnings(test_symbol))
df.head()

##### Company Peers

In [ ]:
# Company Peers
df = pd.DataFrame(finnhub_client.company_peers(test_symbol))
df['symbol'] = test_symbol
df = df[df[0]!=df['symbol']]
df.rename(columns={0:'peer'}, inplace=True)
df.head()

##### Company Profile 2

In [ ]:
# Company Profile 2
df = pd.json_normalize(finnhub_client.company_profile2(symbol=test_symbol))
df.head()

##### List Country

In [ ]:
# List country
df = pd.json_normalize(finnhub_client.country())
df.head()

##### Crypto Exchange

In [ ]:
# Crypto Exchange
df = pd.DataFrame(finnhub_client.crypto_exchanges())
df.head()

##### Crypto Symbols

In [ ]:
# Crypto symbols
df = pd.json_normalize(finnhub_client.crypto_symbols('BINANCE'))
df.head()

##### Financials Reported

In [ ]:
# Financials as reported
df = pd.json_normalize(finnhub_client.financials_reported(symbol=test_symbol, freq='annual'))
## df = pd.json_normalize(df['data']).reset_index()
df = normalize_json_dataframe(pd.DataFrame(df['data']))
df = pd.DataFrame(df[0]).drop(columns=['original_index'])
## report_bs_df = normalize_json_dataframe(df)## pd.DataFrame(df['report.bs']))
## report_bs_df
df.head()

In [ ]:
report_pre_df = df.copy(deep=True)
report_pre_df.rename(columns={'report.bs':'bs', 'report.ic':'ic', 'report.cf':'cf'}, inplace=True)
duckdb.register("report_pre_df_view", report_pre_df)
res = duckdb.query("SELECT \
                   accessNumber, symbol, cik, year, quarter, form, startDate, endDate, filedDate, acceptedDate, \
                   CAST(json_extract(unnest(bs), '$.concept') AS STRING) AS report_bs_concept, \
                   CAST(json_extract(unnest(bs), '$.unit') AS STRING) AS report_bs_unit, \
                   CAST(json_extract(unnest(bs), '$.label') AS STRING) AS report_bs_label, \
                   CAST(json_extract(unnest(bs), '$.value') AS FLOAT) AS report_bs_value, \
                   CAST(json_extract(unnest(ic), '$.concept') AS STRING) AS report_ic_concept, \
                   CAST(json_extract(unnest(ic), '$.unit') AS STRING) AS report_ic_unit, \
                   CAST(json_extract(unnest(ic), '$.label') AS STRING) AS report_ic_label, \
                   CAST(json_extract(unnest(ic), '$.value') AS FLOAT) AS report_ic_value, \
                   CAST(json_extract(unnest(cf), '$.concept') AS STRING) AS report_cf_concept, \
                   CAST(json_extract(unnest(cf), '$.unit') AS STRING) AS report_cf_unit, \
                   CAST(json_extract(unnest(cf), '$.label') AS STRING) AS report_cf_label, \
                   CAST(json_extract(unnest(cf), '$.value') AS FLOAT) AS report_cf_value, \
                   FROM report_pre_df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

##### Forex Exchanges

In [ ]:
# Forex exchanges
df = pd.DataFrame(finnhub_client.forex_exchanges())
df.rename(columns={0:'forex_exch'}, inplace=True)
df.head()

##### Forex Symbols

In [ ]:
# Forex symbols
df = pd.DataFrame(finnhub_client.forex_symbols('OANDA'))
df.head()

##### General News

In [ ]:
# General news
df = pd.DataFrame(finnhub_client.general_news('forex', min_id=0))
df.drop(columns=['image'], inplace=True)
df.head()

##### IPO Calendar

In [ ]:
# IPO calendar
df = pd.DataFrame(finnhub_client.ipo_calendar(_from="2020-05-01", to="2020-06-01"))
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(ipoCalendar, '$.date') AS DATE) AS date, \
                   CAST(json_extract(ipoCalendar, '$.exchange') AS STRING) AS exchange, \
                   CAST(json_extract(ipoCalendar, '$.name') AS STRING) AS name, \
                   CAST(json_extract(ipoCalendar, '$.numberOfShares') AS INT) AS numberOfShares, \
                   CAST(json_extract(ipoCalendar, '$.price') AS FLOAT) AS price, \
                   CAST(json_extract(ipoCalendar, '$.status') AS STRING) AS status, \
                   CAST(json_extract(ipoCalendar, '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(ipoCalendar, '$.totalSharesValue') AS FLOAT) AS totalSharesValue, \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

##### Quote

In [ ]:
# Quote
df = pd.json_normalize(finnhub_client.quote(test_symbol))
df

##### Recommendation Trends

In [ ]:
# Recommendation trends
df = pd.json_normalize(finnhub_client.recommendation_trends(test_symbol))
df

##### Stock Symbols

In [ ]:
# Stock symbols
df = pd.json_normalize(finnhub_client.stock_symbols('US')[0:5])
df

##### Earnings Calendar

In [ ]:
# Earnings Calendar
df = pd.json_normalize(finnhub_client.earnings_calendar(_from="2005-03-16", to="2026-03-16", symbol=test_symbol, international=False))
df

##### Covid-19

In [ ]:
# Covid-19
df = pd.json_normalize(finnhub_client.covid19())
df

##### FDA Calendar

In [ ]:
# FDA Calendar
df = pd.json_normalize(finnhub_client.fda_calendar())
df

##### Symbol Lookup

In [ ]:
# Symbol lookup
df = pd.json_normalize(finnhub_client.symbol_lookup(test_symbol_name))
df

In [ ]:
## symbol_lookup_pre_df = df.copy(deep=True)
df.drop(columns=['count'], inplace=True)
## symbol_lookup_pre_df.rename(columns={'':'','':''}, inplace=True)
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(result), '$.description') AS STRING) AS description, \
                   CAST(json_extract(unnest(result), '$.displaySymbol') AS STRING) AS displaySymbol, \
                   CAST(json_extract(unnest(result), '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(unnest(result), '$.type') AS STRING) AS type \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

##### Visa Application (Not Working as of Last Test)

In [ ]:
# Visa application
df = pd.json_normalize(finnhub_client.stock_visa_application(test_symbol, "2015-01-01", "2022-06-15"))
df

##### Insider Sentiment

In [ ]:
# Insider sentiment
df = pd.json_normalize(finnhub_client.stock_insider_sentiment(test_symbol, '2021-01-01', '2022-03-01'))
df

In [ ]:
df.drop(columns=['symbol'], inplace=True)
duckdb.register("df_view", df)
res = duckdb.query("SELECT \
                   CAST(json_extract(unnest(data), '$.symbol') AS STRING) AS symbol, \
                   CAST(json_extract(unnest(data), '$.year') AS INT) AS year, \
                   CAST(json_extract(unnest(data), '$.month') AS INT) AS month, \
                   CAST(json_extract(unnest(data), '$.change') AS FLOAT) AS change, \
                   CAST(json_extract(unnest(data), '$.mspr') AS FLOAT) AS mspr \
                   FROM df_view").to_df()  ##WHERE period == '1985-09-27'").to_df()
res.replace('"', '', regex=True, inplace=True)
res.head()

##### USA Spending (Not Working as of Last Test)

In [ ]:
# USA Spending
df = pd.json_normalize(finnhub_client.stock_usa_spending(test_symbol, "2021-01-01", "2022-06-15"))
df

##### Market Holiday (Not Working as of Last Test)

In [ ]:
## Market Holday / Status
df = pd.json_normalize(finnhub_client.market_holiday(exchange='US'))
df


##### Market Status (Not Working as of Last Test)

In [ ]:
df = pd.json_normalize(finnhub_client.market_status(exchange='US'))
df

## Paid Endpoints